In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
os.chdir('drive/MyDrive/AI+X고급/식물을 부탁해/kjw')

In [5]:
# ???_dataset_detection
# (1) 드라이브 → Colab으로 복사
!cp "/content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/dataset(detection)/스투키_dataset_detection.zip" "/content/"

# (2) Colab에서 압축 해제
!unzip -q "/content/스투키_dataset_detection.zip" -d "/content/스투키_dataset_detection"

# (3) 압축 파일 삭제 (용량 절약)
!rm "/content/스투키_dataset_detection.zip"

In [7]:
import json
import cv2
import random
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

In [8]:
# 음...이건 그냥 코드 돌릴때 일정하게 나올라고 설정한 거임.
# 1. 파이썬 내장 random 시드 고정
random.seed(42)

# 2. 넘파이 시드 고정
np.random.seed(42)

# 3. TensorFlow 시드 고정
tf.random.set_seed(42)

# 4. 환경 변수 (특히 GPU 연산의 비결정성 제어)
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TF_DETERMINISTIC_OPS"] = "1"  # TF 연산을 결정적으로

In [9]:
plant_name = "스투키"

In [10]:
# YOLOv11 설치
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.0 MB/s eta 0:00:00


In [11]:
import os, glob, csv
import numpy as np
import cv2
import tensorflow as tf
import pandas as pd
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from ultralytics import YOLO

# 경로/설정
DET_WEIGHTS = f"yolo_log/{plant_name}_yolo/detect/train/weights/best.pt"
CLS_WEIGHTS = f"mobilenetv3_log/{plant_name}_mobilenetv3_large_best(final).keras"
SOURCE = f"/content/{plant_name}_dataset_detection/test/images"

OUT_DIR = f"final_test_result"
DRAW_DIR = os.path.join(OUT_DIR, f"{plant_name}_draws")
CROP_DIR = os.path.join(OUT_DIR, f"{plant_name}_crops")
os.makedirs(DRAW_DIR, exist_ok=True)
os.makedirs(CROP_DIR, exist_ok=True)

YOLO_CONF = 0.6   # 낮은 신뢰도 탐지 제외 기준
YOLO_IOU  = 0.45
IMG_SIZE  = 640
CLS_SIZE  = (224, 224)
CLASS_NAMES = ["normal", "overwatered", "underwatered"]

# 모델 로드
detection_model = YOLO(DET_WEIGHTS)
classification_model = tf.keras.models.load_model(CLS_WEIGHTS, compile=False)

# 유틸 함수
def list_images(path):
    if os.path.isdir(path):
        imgs = glob.glob(os.path.join(path, "*.jpg"))
        return sorted(imgs)
    return [path]

def preprocess_for_cls(bgr_img, size=(224,224)):
    rgb = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, size, interpolation=cv2.INTER_AREA)
    x = preprocess_input(rgb.astype(np.float32))
    return np.expand_dims(x, axis=0)

def clip_box(x1, y1, x2, y2, W, H):
    x1 = max(0, min(int(x1), W-1))
    y1 = max(0, min(int(y1), H-1))
    x2 = max(0, min(int(x2), W-1))
    y2 = max(0, min(int(y2), H-1))
    return x1, y1, x2, y2

def get_true_label_from_filename(img_name):
    tokens = img_name.split("-")
    for i in range(1, len(tokens) - 1):
        if tokens[i] in {"1","2","3"} and tokens[i-1].isalpha() and tokens[i+1].isalpha():
            return {"1":"underwatered", "2":"normal", "3":"overwatered"}[tokens[i]]
    raise ValueError(f"GT 클래스를 추출할 수 없음: {img_name}")

# 입력 이미지 목록
image_list = list_images(SOURCE)
assert len(image_list) > 0, f"No images found in {SOURCE}"

# 결과 CSV 준비
csv_path = os.path.join(OUT_DIR, f"{plant_name}_test_result_summary.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as csv_f:
    csv_w = csv.writer(csv_f)
    csv_w.writerow(["image","final_pred_class","final_confidence","true_class","correct","num_leaves"])

    for img_path in image_list:
        img_name = os.path.basename(img_path)
        draw = cv2.imread(img_path)
        if draw is None:
            print(f"[WARN] Cannot read image: {img_path}")
            continue
        H, W = draw.shape[:2]
        true_label = get_true_label_from_filename(img_name)

        # YOLO 탐지
        results = detection_model.predict(source=img_path, imgsz=IMG_SIZE,
                                          conf=YOLO_CONF, iou=YOLO_IOU, verbose=False)
        r = results[0]
        boxes = r.boxes

        if boxes is None or len(boxes) == 0:
            print(f"[INFO] No leaves detected: {img_name}")
            csv_w.writerow([img_name, "no_leaf", 0.0, true_label, 0, 0])
            continue

        leaf_probs = []  # 잎별 softmax 확률 저장

        # 탐지된 잎별 분류
        for i, b in enumerate(boxes):
            conf = float(b.conf[0])
            if conf < YOLO_CONF:
                continue  # 낮은 신뢰도 탐지 무시

            x1, y1, x2, y2 = clip_box(*b.xyxy[0].tolist(), W, H)
            leaf = draw[y1:y2, x1:x2].copy()
            if leaf.size == 0:
                continue

            x = preprocess_for_cls(leaf, CLS_SIZE)
            probs = classification_model.predict(x, verbose=0)[0]
            leaf_probs.append({c: float(probs[i]) for i, c in enumerate(CLASS_NAMES)})

        if len(leaf_probs) == 0:
            csv_w.writerow([img_name, "no_valid_leaf", 0.0, true_label, 0, 0])
            continue

        # 소프트 보팅 (평균 확률)
        avg_probs = {c: np.mean([p[c] for p in leaf_probs]) for c in CLASS_NAMES}
        final_class = max(avg_probs, key=avg_probs.get)
        final_conf = float(avg_probs[final_class])
        correct = int(final_class == true_label)

        csv_w.writerow([img_name, final_class, f"{final_conf:.4f}", true_label, correct, len(leaf_probs)])

# 정확도 계산
df = pd.read_csv(csv_path)
acc = df["correct"].mean() if len(df) else 0.0
print(f"최종 정확도: {acc:.4f} (총 식물 이미지={len(df)})")
print(f"결과 CSV: {csv_path}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
최종 정확도: 1.0000 (총 식물 이미지=1800)
결과 CSV: final_test_result/스투키_test_result_summary.csv


## 관수 주기까지 고려한 FLOW